# Tugas Besar Fundamen Sains Data
## Analisis Audio Features Spotify: Unsupervised & Supervised Learning

Notebook ini mencakup:
1. Data Understanding & Cleaning (EDA)
2. Preprocessing (missing value, encoding, scaling)
3. Dimensionality Reduction (PCA)
4. Unsupervised Learning: K-Means, Hierarchical Clustering, DBSCAN
5. Supervised Learning: Random Forest
6. Evaluasi & Kesimpulan

> Dataset: `SpotifyAudioFeaturesApril2019.csv` (~130rb baris). Kolom: `artist_name, track_id, track_name, acousticness, danceability, duration_ms, energy, instrumentalness, key, liveness, loudness, mode, speechiness, tempo, time_signature, valence, popularity`.
>
> **Catatan penting:** dataset ini **tidak punya kolom genre/kategori**. Karena itu, target supervised learning dibuat dari kolom `popularity` (skor 0-100) yang dikelompokkan jadi 3 kategori (`Low`/`Medium`/`High`) memakai *quantile binning* supaya jumlah kelasnya seimbang (lihat Bagian 1.1). Kolom `popularity` sendiri **tidak diikutkan sebagai fitur** setelah itu, karena kalau diikutkan, model jadi bisa "curang" langsung menebak dari nilai yang dipakai membuat labelnya sendiri (data leakage).

## 0. Konfigurasi & Import Library

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.decomposition import PCA
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.cluster import KMeans, AgglomerativeClustering, DBSCAN
from sklearn.metrics import silhouette_score, davies_bouldin_score
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score, classification_report, confusion_matrix,
    ConfusionMatrixDisplay
)
from scipy.cluster.hierarchy import dendrogram, linkage

sns.set_style("whitegrid")
plt.rcParams["figure.figsize"] = (8, 5)

RANDOM_STATE = 42

In [ ]:
# ==== KONFIGURASI ====
DATA_PATH = "data/SpotifyAudioFeaturesApril2019.csv"
TARGET_COL = "popularity_category"  # kolom label hasil turunan dari 'popularity' (lihat Bagian 1.1)
ID_COLS = ["artist_name", "track_id", "track_name", "popularity"]
# 'popularity' ikut di-drop dari fitur karena itu sumber pembuatan label (hindari data leakage)

## 1. Data Understanding & Cleaning (EDA)

In [ ]:
df = pd.read_csv(DATA_PATH)
print(df.shape)
print(df.columns.tolist())
df.head()

### 1.1 Membuat Kolom Target (`popularity_category`)

`popularity` adalah skor kontinu 0-100 dan biasanya sangat miring (banyak lagu bernilai rendah). Memakai *quantile binning* (`pd.qcut`) memastikan 3 kategori punya jumlah data yang relatif seimbang, dibanding pembagian rentang tetap (0-33/34-66/67-100) yang bisa timpang.

In [ ]:
df["popularity_category"] = pd.qcut(
    df["popularity"], q=3, labels=["Low", "Medium", "High"], duplicates="drop"
).astype(str)

print(df["popularity_category"].value_counts())
df["popularity_category"].value_counts().plot(kind="bar")
plt.title("Distribusi Kategori Popularitas")
plt.ylabel("Jumlah lagu")
plt.show()

In [ ]:
df.info()

In [ ]:
print("Missing value per kolom:")
print(df.isna().sum().sort_values(ascending=False).head(15))
print("\nJumlah baris duplikat:", df.duplicated().sum())

In [ ]:
df.describe().T

In [ ]:
numeric_df = df.select_dtypes(include=[np.number]).drop(columns=["popularity"])
plt.figure(figsize=(10, 8))
sns.heatmap(numeric_df.corr(), cmap="coolwarm", center=0, annot=True, fmt=".2f")
plt.title("Heatmap Korelasi Fitur Audio")
plt.show()

## 2. Preprocessing

In [ ]:
clean_df = df.drop(columns=[c for c in ID_COLS if c in df.columns]).copy()

num_cols = clean_df.select_dtypes(include=[np.number]).columns.tolist()
for col in num_cols:
    clean_df[col] = clean_df[col].fillna(clean_df[col].median())

clean_df = clean_df.dropna(subset=[TARGET_COL])
clean_df.shape

In [ ]:
feature_cols = [c for c in num_cols if c != TARGET_COL]
X = clean_df[feature_cols]

le = LabelEncoder()
y = le.fit_transform(clean_df[TARGET_COL])

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)
X_scaled = pd.DataFrame(X_scaled, columns=feature_cols)
X_scaled.head()

## 3. Dimensionality Reduction (PCA)

In [ ]:
pca_full = PCA(random_state=RANDOM_STATE).fit(X_scaled)
cum_var = np.cumsum(pca_full.explained_variance_ratio_)

plt.plot(range(1, len(cum_var) + 1), cum_var, marker="o")
plt.axhline(0.9, color="red", linestyle="--", label="90% variansi")
plt.xlabel("Jumlah Komponen")
plt.ylabel("Kumulatif Explained Variance")
plt.title("Scree Plot PCA")
plt.legend()
plt.show()

n_components_90 = np.argmax(cum_var >= 0.90) + 1
print(f"Jumlah komponen untuk >=90% variansi: {n_components_90}")

In [ ]:
pca_2d = PCA(n_components=2, random_state=RANDOM_STATE)
X_pca_2d = pca_2d.fit_transform(X_scaled)

pca_reduced = PCA(n_components=n_components_90, random_state=RANDOM_STATE)
X_pca_reduced = pca_reduced.fit_transform(X_scaled)

print("Shape data setelah PCA (2D):", X_pca_2d.shape)
print("Shape data setelah PCA (reduced):", X_pca_reduced.shape)

## 4. Unsupervised Learning: Clustering

> Catatan: dataset ini besar (~130rb baris). Kalau proses clustering terasa berat/lambat, ambil sampel dulu (mis. `X_pca_reduced_sample = X_pca_reduced[np.random.RandomState(RANDOM_STATE).choice(len(X_pca_reduced), 20000, replace=False)]`) sebelum fitting K-Means/Hierarchical/DBSCAN.

### 4.1 K-Means

In [ ]:
inertias, sil_scores = [], []
k_range = range(2, 11)

for k in k_range:
    km = KMeans(n_clusters=k, random_state=RANDOM_STATE, n_init=10)
    labels = km.fit_predict(X_pca_reduced)
    inertias.append(km.inertia_)
    sil_scores.append(silhouette_score(X_pca_reduced, labels))

fig, ax = plt.subplots(1, 2, figsize=(14, 4))
ax[0].plot(k_range, inertias, marker="o")
ax[0].set_title("Elbow Method")
ax[0].set_xlabel("k")
ax[0].set_ylabel("Inertia")

ax[1].plot(k_range, sil_scores, marker="o", color="green")
ax[1].set_title("Silhouette Score")
ax[1].set_xlabel("k")
ax[1].set_ylabel("Silhouette Score")
plt.show()

In [ ]:
K_OPTIMAL = 4  # TODO: ganti sesuai hasil elbow/silhouette

kmeans_final = KMeans(n_clusters=K_OPTIMAL, random_state=RANDOM_STATE, n_init=10)
kmeans_labels = kmeans_final.fit_predict(X_pca_reduced)

plt.scatter(X_pca_2d[:, 0], X_pca_2d[:, 1], c=kmeans_labels, cmap="tab10", s=5, alpha=0.5)
plt.title(f"K-Means Clustering (k={K_OPTIMAL}) - proyeksi PCA 2D")
plt.xlabel("PC1")
plt.ylabel("PC2")
plt.show()

### 4.2 Hierarchical Clustering

In [ ]:
sample_size = min(200, X_pca_reduced.shape[0])
sample_idx = np.random.RandomState(RANDOM_STATE).choice(
    X_pca_reduced.shape[0], sample_size, replace=False
)
X_sample = X_pca_reduced[sample_idx]

linked = linkage(X_sample, method="ward")
plt.figure(figsize=(14, 5))
dendrogram(linked, truncate_mode="lastp", p=30)
plt.title("Dendrogram (Ward Linkage, sampel data)")
plt.xlabel("Sampel")
plt.ylabel("Jarak")
plt.show()

In [ ]:
# Hierarchical clustering di-fit pada sampel juga (data terlalu besar untuk seluruh baris)
HIER_SAMPLE_SIZE = min(5000, X_pca_reduced.shape[0])
hier_idx = np.random.RandomState(RANDOM_STATE).choice(
    X_pca_reduced.shape[0], HIER_SAMPLE_SIZE, replace=False
)
X_hier_sample = X_pca_reduced[hier_idx]
X_pca_2d_hier_sample = X_pca_2d[hier_idx]

agglo = AgglomerativeClustering(n_clusters=K_OPTIMAL, linkage="ward")
agglo_labels = agglo.fit_predict(X_hier_sample)

plt.scatter(X_pca_2d_hier_sample[:, 0], X_pca_2d_hier_sample[:, 1], c=agglo_labels, cmap="tab10", s=5, alpha=0.5)
plt.title(f"Hierarchical Clustering (k={K_OPTIMAL}) - sampel {HIER_SAMPLE_SIZE} baris")
plt.xlabel("PC1")
plt.ylabel("PC2")
plt.show()

### 4.3 DBSCAN

In [ ]:
from sklearn.neighbors import NearestNeighbors

MIN_SAMPLES = 5
# k-distance plot juga dihitung dari sampel agar cepat
kdist_idx = np.random.RandomState(RANDOM_STATE).choice(
    X_pca_reduced.shape[0], min(10000, X_pca_reduced.shape[0]), replace=False
)
neighbors = NearestNeighbors(n_neighbors=MIN_SAMPLES).fit(X_pca_reduced[kdist_idx])
distances, _ = neighbors.kneighbors(X_pca_reduced[kdist_idx])
k_distances = np.sort(distances[:, -1])

plt.plot(k_distances)
plt.title("K-Distance Plot (untuk menentukan eps)")
plt.xlabel("Titik data (terurut)")
plt.ylabel(f"Jarak ke tetangga ke-{MIN_SAMPLES}")
plt.show()

In [ ]:
EPS = 0.5  # TODO: ganti sesuai titik "siku" pada k-distance plot
DBSCAN_SAMPLE_SIZE = min(20000, X_pca_reduced.shape[0])
dbscan_idx = np.random.RandomState(RANDOM_STATE).choice(
    X_pca_reduced.shape[0], DBSCAN_SAMPLE_SIZE, replace=False
)
X_dbscan_sample = X_pca_reduced[dbscan_idx]
X_pca_2d_dbscan_sample = X_pca_2d[dbscan_idx]

dbscan = DBSCAN(eps=EPS, min_samples=MIN_SAMPLES)
dbscan_labels = dbscan.fit_predict(X_dbscan_sample)

n_clusters = len(set(dbscan_labels)) - (1 if -1 in dbscan_labels else 0)
n_noise = list(dbscan_labels).count(-1)
print(f"Jumlah cluster: {n_clusters}, jumlah noise: {n_noise}")

plt.scatter(X_pca_2d_dbscan_sample[:, 0], X_pca_2d_dbscan_sample[:, 1], c=dbscan_labels, cmap="tab10", s=5, alpha=0.5)
plt.title("DBSCAN Clustering - proyeksi PCA 2D (label -1 = noise)")
plt.xlabel("PC1")
plt.ylabel("PC2")
plt.show()

### 4.4 Perbandingan Evaluasi Clustering

> Catatan: karena K-Means dihitung dari seluruh data sedangkan Hierarchical & DBSCAN dari sampel (alasan performa), evaluasi silhouette/davies-bouldin di bawah dihitung ulang pada sampel yang sama besarnya (`HIER_SAMPLE_SIZE`) supaya perbandingannya adil.

In [ ]:
def safe_eval(X, labels, name):
    labels = np.asarray(labels)
    mask = labels != -1
    if len(set(labels[mask])) < 2:
        return {"Metode": name, "Silhouette": None, "Davies-Bouldin": None}
    return {
        "Metode": name,
        "Silhouette": silhouette_score(X[mask], labels[mask]),
        "Davies-Bouldin": davies_bouldin_score(X[mask], labels[mask]),
    }

kmeans_labels_on_hier_sample = kmeans_final.predict(X_hier_sample)

results = [
    safe_eval(X_hier_sample, kmeans_labels_on_hier_sample, "K-Means"),
    safe_eval(X_hier_sample, agglo_labels, "Hierarchical"),
    safe_eval(X_dbscan_sample, dbscan_labels, "DBSCAN"),
]
pd.DataFrame(results)

## 5. Supervised Learning: Random Forest

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X_scaled, y, test_size=0.2, random_state=RANDOM_STATE, stratify=y
)

rf = RandomForestClassifier(
    n_estimators=300,
    random_state=RANDOM_STATE,
    class_weight="balanced",
    n_jobs=-1,
)
rf.fit(X_train, y_train)

y_pred = rf.predict(X_test)
print("Akurasi:", accuracy_score(y_test, y_pred))
print("\nClassification Report:\n")
print(classification_report(y_test, y_pred, target_names=le.classes_))

In [ ]:
cm = confusion_matrix(y_test, y_pred)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=le.classes_)
fig, ax = plt.subplots(figsize=(6, 6))
disp.plot(ax=ax, colorbar=False)
plt.title("Confusion Matrix - Random Forest")
plt.show()

In [ ]:
importances = pd.Series(rf.feature_importances_, index=feature_cols)
importances.sort_values(ascending=False).plot(kind="barh")
plt.title("Feature Importance - Random Forest")
plt.gca().invert_yaxis()
plt.show()

## 6. Kesimpulan

Isi bagian ini dengan interpretasi hasil, misalnya:
- Berapa komponen PCA yang cukup, dan fitur audio apa yang paling dominan di tiap komponen.
- Metode clustering mana yang paling koheren, dan karakteristik tiap cluster (mis. cluster "energik & danceable" vs "akustik & tenang").
- Seberapa baik Random Forest memprediksi kategori popularitas dari fitur audio saja (tanpa tahu popularity aslinya) — kalau akurasinya rendah/sedang, itu insight yang valid: audio feature saja mungkin tidak cukup untuk menjelaskan popularitas (faktor lain seperti marketing, artis, era rilis, dsb juga berperan).
- Fitur audio paling berpengaruh terhadap popularitas (lihat feature importance).